# 02 — NumPy Fundamentals: Thinking in Shapes

**Overview:** ndarray creation, shape/dtype, indexing/slicing/masking, reshape/transpose,
concatenate/stack, broadcasting, vectorization, dot/matmul, reductions, and tensor-shape
reasoning through a mini linear-layer computation.

**Required libraries:** `numpy`.

```
!pip install -q numpy   # already present on Colab
```

Connects to: `01_python` (the `ToyDataset` class becomes an `ndarray`-backed structure). Leads
to: `03_data_libraries` (Pandas/Polars DataFrames are built on this same shape-thinking).

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **ReLU** | Rectified Linear Unit | Activation max(0, x). |

## 0. Environment

In [1]:
import sys, numpy as np
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
rng = np.random.default_rng(42)   # deterministic seed


Python: 3.12.5
NumPy: 2.5.3


## 1. Creating arrays & inspecting shape/dtype/ndim

💡 **Key Idea:** `shape` is a tuple — its length is `ndim`, and each entry is the size along
that axis.

In [2]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([[1, 2, 3], [4, 5, 6]])
c = rng.normal(size=(2, 3, 4))   # 3D tensor

for name, arr in [("a", a), ("b", b), ("c", c)]:
    print(f"{name}: shape={arr.shape}, ndim={arr.ndim}, dtype={arr.dtype}")

assert a.shape == (5,)
assert b.shape == (2, 3)
assert c.shape == (2, 3, 4)


a: shape=(5,), ndim=1, dtype=int64
b: shape=(2, 3), ndim=2, dtype=int64
c: shape=(2, 3, 4), ndim=3, dtype=float64


## 2. Indexing, slicing, boolean masking, fancy indexing

In [3]:
x = np.arange(12).reshape(3, 4)
print("x:\n", x)
print("x[1, 2]:", x[1, 2])          # single element
print("x[:, 1]:", x[:, 1])          # column 1, shape (3,)
print("x[1, :]:", x[1, :])          # row 1, shape (4,)
print("x[x > 5]:", x[x > 5])        # boolean mask -> 1D array of matches
print("x[[0, 2]]:\n", x[[0, 2]])    # fancy indexing -> rows 0 and 2

assert x[:, 1].shape == (3,)
assert x[1, :].shape == (4,)
assert list(x[x > 5]) == [6, 7, 8, 9, 10, 11]


x:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
x[1, 2]: 6
x[:, 1]: [1 5 9]
x[1, :]: [4 5 6 7]
x[x > 5]: [ 6  7  8  9 10 11]
x[[0, 2]]:
 [[ 0  1  2  3]
 [ 8  9 10 11]]


## 3. reshape, transpose, squeeze, expand_dims, concatenate, stack

⚠️ **Common Mistake:** `.reshape()` often returns a **view**, not a copy — mutating it can
mutate the original array.

In [4]:
y = np.arange(6)
y_reshaped = y.reshape(2, 3)
print("original y:", y)
print("reshaped (2,3):\n", y_reshaped)

y_reshaped[0, 0] = 999
print("after mutating the reshaped view, original y:", y)   # also changed!
assert y[0] == 999

z = np.arange(6).reshape(2, 3)
print("z.T shape:", z.T.shape)                 # transpose: (2,3) -> (3,2)
assert z.T.shape == (3, 2)

w = np.array([[[1, 2, 3]]])                    # shape (1, 1, 3)
print("w.shape:", w.shape, "squeezed:", np.squeeze(w).shape)
assert np.squeeze(w).shape == (3,)

v = np.array([1, 2, 3])
print("expand_dims axis=0:", np.expand_dims(v, axis=0).shape)   # (1, 3)
print("expand_dims axis=1:", np.expand_dims(v, axis=1).shape)   # (3, 1)

p = np.array([[1, 2], [3, 4]])
q = np.array([[5, 6]])
print("concatenate axis=0:\n", np.concatenate([p, q], axis=0))   # stacks rows -> (3, 2)
print("stack creates a NEW axis:", np.stack([p, p]).shape)        # (2, 2, 2)
assert np.concatenate([p, q], axis=0).shape == (3, 2)
assert np.stack([p, p]).shape == (2, 2, 2)


original y: [0 1 2 3 4 5]
reshaped (2,3):
 [[0 1 2]
 [3 4 5]]
after mutating the reshaped view, original y: [999   1   2   3   4   5]
z.T shape: (3, 2)
w.shape: (1, 1, 3) squeezed: (3,)
expand_dims axis=0: (1, 3)
expand_dims axis=1: (3, 1)
concatenate axis=0:
 [[1 2]
 [3 4]
 [5 6]]
stack creates a NEW axis: (2, 2, 2)


## 4. Broadcasting

💡 **Key Idea:** two shapes are broadcast-compatible if, comparing dimensions from the right,
each pair is either equal or one of them is 1. Here `(32, 10) + (10,)` works, but
`(32, 10) + (5,)` doesn't.

In [5]:
batch = rng.normal(size=(32, 10))          # 32 samples, 10 features
feature_bias = rng.normal(size=(10,))      # one bias per feature

result = batch + feature_bias               # broadcasts (10,) across all 32 rows
print("batch.shape:", batch.shape, "feature_bias.shape:", feature_bias.shape, "result.shape:", result.shape)
assert result.shape == (32, 10)

try:
    bad = batch + rng.normal(size=(5,))      # incompatible: last dims 10 vs 5
except ValueError as e:
    print("expected broadcasting error:", e)


batch.shape: (32, 10) feature_bias.shape: (10,) result.shape: (32, 10)
expected broadcasting error: operands could not be broadcast together with shapes (32,10) (5,) 


## 5. Vectorization vs Python loops

🔬 **Experiment:** compare a pure-Python loop to a vectorized NumPy op.

In [6]:
import time

n = 2_000_000
data = list(range(n))
arr = np.arange(n)

start = time.perf_counter()
loop_result = [x * x for x in data]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vec_result = arr ** 2
vec_time = time.perf_counter() - start

print(f"pure-Python loop: {loop_time:.4f}s")
print(f"vectorized NumPy: {vec_time:.4f}s")
print(f"speedup: {loop_time / vec_time:.1f}x")

assert list(vec_result[:5]) == loop_result[:5]
assert vec_time < loop_time


pure-Python loop: 0.0334s
vectorized NumPy: 0.0103s
speedup: 3.2x


## 6. dot product, matrix multiplication, reductions

In [7]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
print("dot product:", np.dot(u, v), "==", (1*4 + 2*5 + 3*6))
assert np.dot(u, v) == 32

M = rng.normal(size=(4, 3))
N = rng.normal(size=(3, 2))
P = M @ N                       # matrix multiplication
print("M @ N shape:", P.shape)  # (4, 2)
assert P.shape == (4, 2)

X = rng.normal(size=(5, 4))
print("sum all:", X.sum())
print("sum axis=0 (collapse rows, per-column totals) shape:", X.sum(axis=0).shape)   # (4,)
print("sum axis=1 (collapse columns, per-row totals) shape:", X.sum(axis=1).shape)   # (5,)
print("argmax axis=1 (index of max per row):", X.argmax(axis=1))

assert X.sum(axis=0).shape == (4,)
assert X.sum(axis=1).shape == (5,)


dot product: 32 == 32
M @ N shape: (4, 2)
sum all: -5.52933643985421
sum axis=0 (collapse rows, per-column totals) shape: (4,)
sum axis=1 (collapse columns, per-row totals) shape: (5,)
argmax axis=1 (index of max per row): [1 0 2 2 3]


## 7. Tensor-shape reasoning: a mini linear layer

🎯 **Interview Tip:** `Y = X @ W + b` is the entire forward pass of `nn.Linear`. Being able to
state each dimension's meaning instantly is one of the most common whiteboard checks.

In [8]:
batch_size, in_features, out_features = 32, 10, 3

X = rng.normal(size=(batch_size, in_features))   # (32, 10): 32 samples, 10 features each
W = rng.normal(size=(in_features, out_features))  # (10, 3): maps 10 features -> 3 outputs
b = rng.normal(size=(out_features,))              # (3,): one bias per output unit

Y = X @ W + b   # (32,10)@(10,3) -> (32,3), then broadcast-add (3,)
print("X:", X.shape, "W:", W.shape, "b:", b.shape, "-> Y:", Y.shape)

assert Y.shape == (batch_size, out_features)
print("Interpretation: Y[i, j] = the j-th output unit's value for the i-th sample in the batch.")


X: (32, 10) W: (10, 3) b: (3,) -> Y: (32, 3)
Interpretation: Y[i, j] = the j-th output unit's value for the i-th sample in the batch.


## 8. Connection to what's next

A Pandas/Polars `DataFrame` is conceptually a 2D array with named, typed columns — the same
shape-thinking applies once labels are added. Next: `03_data_libraries`.

## 9. Interview questions
1. State NumPy's broadcasting compatibility rule from memory.
2. Why is `arr ** 2` faster than a Python list comprehension doing the same thing?
3. What's the practical risk of `.reshape()` returning a view instead of a copy?
4. For `X: (32, 10) @ W: (10, 3)`, what does each of the resulting shape's two dimensions mean?
5. What's the difference between `axis=0` and `axis=1` for a reduction on a 2D array?

## 10. Exercises
1. Create a `(5, 3)` random array; compute per-axis means and verify shapes `(3,)` and `(5,)`.
2. Implement vectorized ReLU using boolean masking (no loops).
3. Given `X: (100, 20)`, `W: (20, 5)`, `b: (5,)`, compute and assert the shape of `X @ W + b`.
4. Compute the pairwise squared Euclidean distance matrix for a `(50, 8)` array using only broadcasting (no loops).

## 11. Summary

✅ `ndarray` shape/dtype, indexing/broadcasting/vectorization, and matrix multiplication are the
entire vocabulary behind every neural network's forward pass. Next: `03_data_libraries` for
tabular data at scale.